# Session 5 - Lab: one space for photos and sentences, and what you can ask it

**DAT09-06 Representation Learning · Tuesday 17 November 2026 · 1 h 20**

Yesterday the matching game put two shots of the same product close together.
Today the second shot is a **sentence**: the product's title. You train two
small **heads** that bring a frozen photo expert and a frozen sentence expert
into **one shared space**, you type a request and get products back, and you
find out what that space can be asked - and what it cannot.

### The words of the lecture, and the names your AI will use

| In the lecture | In papers and in code |
|---|---|
| the **photo expert**, the **sentence expert** - both frozen | pretrained *encoders*, or *towers*: here CLIP's image tower and MiniLM |
| a **head**: one small layer on top of an expert | a *projection head*, an *adapter* |
| the **shared space** | a *joint embedding space* |
| the **matching game**, played on **the grid** | the *contrastive loss* (InfoNCE, the CLIP loss) on the *similarity matrix* |
| **found in the top k** | *recall@k* |
| **naming** a product without training a classifier | *zero-shot classification*; the sentences are *prompts* |

### How this notebook is marked

| Marker | What it means |
|---|---|
| `# Given - read it, run it, move on.` | nothing to write. Run the cell. |
| ✏️ **YOUR TURN** | a boxed banner tells you **how many lines** and **what they do**. That is the only place you write code. |
| ❓ **Checkpoint** | **three questions** about *your* output. Answer **in writing**, in the cell below. Most only ask you to read what you just printed. |
| ⭐ **Going further** | optional, harder. Only if you are ahead: the next checkpoint never needs them. |

There are **8 cells to write**, none longer than five lines. Everything else is
given, including the training loop and every plot.

### Read this before you start

An AI will write all eight in two minutes. Let it - but **read what it
prints**: the trap of the day is not a bug in the code. It is a score of
100 % that is perfectly correct and means nothing. You meet it on purpose in
checkpoint 2.

The questions are the lab. The code is the easy part.

### Six checkpoints

| | What you produce | ~ |
|---|---|---|
| **1** | Two experts and no common language: the score to beat | 15 min |
| **2** | The matching game with sentences: trained heads, and the trap | 20 min |
| **3** | Type a sentence, get products - and three requests it cannot serve | 15 min |
| **4** | **The sentences are the instruction** - four sets of sentences, one table | 20 min |
| **5** | How many pairs, and which temperature | 10 min |
| **6** | **The challenge** - the best search that can still name a product | open |

**Most of you will not reach checkpoint 6. That is deliberate.**

Run the cells **in order**.

---

## 0 · Setup

Four given cells: imports, the catalogue, a helper to show products, and the
sentence expert.

In [ ]:
# Given - read it, run it, move on.
import logging
import os
import sys
import textwrap
import time
from pathlib import Path

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # a missing font is not news
os.environ.setdefault("HF_HUB_VERBOSITY", "error")                     # nor is an anonymous download
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

sys.path.insert(0, str(Path.cwd().parent))   # make `common` importable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

from common.evaluation import evaluate_representation
from common.paths import PROCESSED, require
from common.plotting import PALETTE, use_course_style
from common.seeding import SEED, cpu_threads, set_seed

use_course_style()
set_seed(SEED)
cpu_threads()

print("torch", torch.__version__, "| device: cpu")

### 0.2 · The catalogue

6,000 real products from an online fashion shop, twelve kinds, 500 of each.
Every product has a **photo** and a **title**.

Nobody runs a vision model today. Each photo was read **once**, before the
course, by a frozen **photo expert** (the image half of CLIP): 512 numbers per
photo. Each title was read once by a frozen **sentence expert** (MiniLM): 384
numbers per title. Those two tables are all you load.

4,800 products are for training. **1,200 are held out**, 100 of each kind:
every score we believe is computed on those.

In [ ]:
# Given - read it, run it, move on.
HINT = "git pull   (the product files ship with the repository)"

meta = pd.read_parquet(require(PROCESSED / "products_meta.parquet", hint=HINT))


def load_codes(name):
    """A table of frozen codes, one row per product, every row scaled to length 1."""
    table = np.load(require(PROCESSED / name, hint=HINT)).astype(np.float32)
    return F.normalize(torch.from_numpy(table), dim=1)


photo_codes = load_codes("products_image_emb.npy")     # (6000, 512)  the photo expert
title_codes = load_codes("products_text_emb.npy")      # (6000, 384)  the sentence expert

train = torch.tensor((meta["split"] == "train").to_numpy())
held = ~train

kinds = sorted(meta["article_type"].unique())
kind_of = torch.tensor([kinds.index(k) for k in meta["article_type"]])

print(f"photo codes {tuple(photo_codes.shape)}   title codes {tuple(title_codes.shape)}")
print(f"train {int(train.sum())}   held out {int(held.sum())}")
print("kinds:", ", ".join(kinds))

In [ ]:
# Given - read it, run it, move on.
THUMBS = PROCESSED / "products_images"


def show_products(rows, title=None, captions=None, frame=None):
    """Show the photos of these rows of the catalogue, with their titles underneath."""
    rows = [int(r) for r in rows]
    fig, axes = plt.subplots(1, len(rows), figsize=(len(rows) * 1.45, 2.5))
    for c, (ax, r) in enumerate(zip(np.atleast_1d(axes), rows)):
        ax.imshow(Image.open(THUMBS / f"{meta['id'][r]}.jpg"))
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        text = captions[c] if captions is not None else meta["title"][r]
        ax.set_xlabel(textwrap.fill(str(text), 17), fontsize=7)
        if frame is not None and r == int(frame):
            for side in ax.spines.values():
                side.set_visible(True); side.set_color(PALETTE["accent"]); side.set_linewidth(2.5)
    if title:
        fig.suptitle(title, y=1.04, fontsize=11)
    plt.show()


some = torch.where(held)[0][[0, 100, 215, 330, 470, 610, 880, 1100]]
show_products(some, "eight products of the catalogue")

### 0.4 · The sentence expert, for the sentences *you* type

The titles are already encoded. A request you type is not: it has to go
through the same sentence expert. The cell tries to load it (90 MB, once).

If the download fails in class, nothing breaks: **forty requests were encoded
in advance**, and those keep working. `list(CANNED)` shows them.

In [ ]:
# Given - read it, run it, move on.
blob = np.load(require(PROCESSED / "products_queries.npz", hint=HINT))
CANNED = {str(q): torch.from_numpy(e) for q, e in zip(blob["queries"], blob["query_emb"])}

try:
    from sentence_transformers import SentenceTransformer
    sentence_expert = SentenceTransformer("all-MiniLM-L6-v2")
    LIVE = True
except Exception as error:                       # no network, no cache: fall back
    sentence_expert, LIVE = None, False


def sentence_code(sentence):
    """The sentence expert's 384 numbers for one sentence, scaled to length 1."""
    if sentence in CANNED:
        code = CANNED[sentence]
    elif LIVE:
        code = torch.from_numpy(sentence_expert.encode([sentence]))[0]
    else:
        raise ValueError("The sentence expert is not loaded on this machine, so only the "
                         "40 prepared requests work. Run list(CANNED) to see them.")
    return F.normalize(code, dim=0)


print("sentence expert:", "loaded - type any request" if LIVE
      else "NOT loaded - use the 40 prepared requests, list(CANNED)")
print("for example:", ", ".join(f'"{q}"' for q in list(CANNED)[10:14]))

---

# Checkpoint 1 - Two experts, no common language

**~15 minutes.** One cell to write, three lines.

### 1.1 ▶ · Each expert is good - alone

Take one product. Ask the photo expert for the five products whose **photo**
code is closest; ask the sentence expert for the five whose **title** code is
closest. Then the number you have used since the autoencoder lab, for both:
is a product of the same kind as its nearest neighbours?

In [ ]:
# Given - read it, run it, move on.
def nearest(codes, row, k=5):
    """The k rows whose code is most similar to the code of `row` (itself excluded)."""
    sim = codes @ codes[row]
    sim[row] = -1.0
    return sim.topk(k).indices


row = int(torch.where(held)[0][330])
show_products([row], "one product")
show_products(nearest(photo_codes, row), "its 5 nearest neighbours for the photo expert")
print("its 5 nearest neighbours for the sentence expert:")
for r in nearest(title_codes, row):
    print("   ", meta["title"][int(r)])

for name, codes in (("photo expert", photo_codes), ("sentence expert", title_codes)):
    rep = evaluate_representation(codes[held].numpy(), kind_of[held].numpy(),
                                  name=name, verbose=False)
    print(f"{name:<16s} k-NN accuracy on the kind of product: {rep.knn_accuracy:.3f}")

### 1.2 ✏️ · The score of the day: found in the top k

A customer types a sentence; the search ranks every photo; we ask whether the
**right** photo is among the first *k*.

Row *i* of `queries` is looking for row *i* of `gallery`. Build the grid of
similarities (one row per query, one column per gallery item). The right
answers are on the **diagonal**. A query's right answer is in its top *k* when
**fewer than *k*** wrong answers score higher than it.

The check uses four queries and four gallery items you can draw on paper:
codes of 2 numbers, given as angles.

*(Papers call this score recall@k. Yesterday's "found again" was k = 1.)*

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 3 lines
#     sim: the grid, queries @ gallery.T
#     how many entries of each row are larger than its diagonal entry
#     return the share of rows where that count is below k
# ──────────────────────────────────────────────────────────────────────
def found_in_top(queries, gallery, k=10):
    """Share of queries whose own gallery item (same row) is among their k most similar."""
    queries, gallery = F.normalize(queries, dim=1), F.normalize(gallery, dim=1)
    ...          # <- replace this line


def at_angles(degrees):
    a = torch.deg2rad(torch.tensor(degrees, dtype=torch.float32))
    return torch.stack([a.cos(), a.sin()], dim=1)


toy_gallery = at_angles([0, 30, 90, 180])
toy_queries = at_angles([5, 70, 80, 170])        # the second one is nearer 90 than its own 30
print(f"top 1: {found_in_top(toy_queries, toy_gallery, k=1):.2f} (expect 0.75)   "
      f"top 2: {found_in_top(toy_queries, toy_gallery, k=2):.2f} (expect 1.00)")

### 1.3 ▶ · Try to compare the two experts

First directly. Then through two **heads**: one small layer per expert, both
ending in the same 64 numbers - for now with the random numbers they are born
with.

In [ ]:
# Given - read it, run it, move on.
try:
    title_codes[held] @ photo_codes[held].T
except RuntimeError as error:
    print("title codes x photo codes, directly:", str(error).split("(")[0].strip())


class Heads(nn.Module):
    """One small layer per expert, both ending in the same d numbers: the shared space."""

    def __init__(self, d=64):
        super().__init__()
        self.photo = nn.Linear(512, d)      # photo codes    -> shared space
        self.text = nn.Linear(384, d)       # sentence codes -> shared space


set_seed(SEED)
untrained = Heads()
with torch.no_grad():
    score = found_in_top(untrained.text(title_codes[held]), untrained.photo(photo_codes[held]))
print(f"untrained heads: found in the top 10 = {score:.3f}"
      f"   (pure chance: 10 out of 1,200 = {10 / 1200:.3f})")

### ❓ Checkpoint 1

1. How many numbers does each expert use for one product? In 1.3, what happens
   when you multiply the title codes by the photo codes directly?
2. In 1.1, are the photo expert's five neighbours the same kind of product as
   the one you asked about? Are the sentence expert's five titles?
3. With untrained heads, what share of the 1,200 held-out titles find their own
   photo in the top 10? What would pure chance give?

**⭐ Going further - optional**

4. In the toy check, one query misses the top 1. Which one, and which gallery
   item beats its own? *(Draw the eight angles.)*

*Your answers:*

1.
2.
3.
⭐ 4.

---

# Checkpoint 2 - The matching game, with sentences

**~20 minutes.** Two cells to write.

Yesterday: two shots of the same product, and each shot A had to pick its own
shot B among the N of the batch. Today the second shot is **the product's
title**. Nothing else changes.

### 2.1 ✏️ · The same function as yesterday

Yesterday you wrote it in two steps, the grid and then the loss. Here it is one
function, with the same name, the same arguments and the same result.

`a` and `b` are two tables of N codes; row *i* of `a` goes with row *i* of `b`.

- scale every code to length 1;
- **the grid**: N × N similarities, divided by the temperature;
- the right answer of row *i* is column *i*;
- play the game **along the rows** (each photo picks its sentence) and **down
  the columns** (each sentence picks its photo): `F.cross_entropy` twice,
  averaged.

*(Papers: the contrastive loss, InfoNCE; with a photo and a sentence, the CLIP
loss.)*

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 4 lines
#     scale a and b to length 1 (F.normalize, dim=1)
#     grid = a @ b.T / temperature
#     target = torch.arange(len(a))
#     cross-entropy of grid AND of grid.T against target, averaged
# ──────────────────────────────────────────────────────────────────────
def matching_loss(a, b, temperature=0.1):
    """The matching game on N pairs: row i of a goes with row i of b."""
    ...          # <- replace this line


set_seed(0)
a = torch.randn(256, 64)
print(f"256 pairs that match perfectly : {matching_loss(a, a).item():.2f}   (expect about 0.03)")
print(f"256 pairs with no link at all  : {matching_loss(a, torch.randn(256, 64)).item():.2f}"
      f"   (expect about 6.3 - picking among 256 at random would give log(256) = 5.55)")

### 2.2 ✏️ · One round of the game

A batch of photo codes (512 numbers each) and the sentence codes of the same
products (384 numbers each). Send each through its head, and play.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 1 line
#     photos through heads.photo, sentences through heads.text,
#     then matching_loss on the two results, at this temperature
# ──────────────────────────────────────────────────────────────────────
def game_step(heads, photos, sentences, temperature=0.1):
    """The matching loss of one batch of (photo code, sentence code) pairs."""
    ...          # <- replace this line

### 2.3 · The training loop

Given, and you will not touch it again. The two experts stay frozen: the only
numbers that move are the two heads, about 57,000 of them.

In [ ]:
# Given - read it, run it, move on.
def train_heads(photos, sentences, *, d=64, epochs=20, batch=256, temperature=0.1,
                lr=1e-3, verbose=False):
    """Train two heads on (photo code, sentence code) pairs. Returns the heads."""
    set_seed(SEED)
    heads = Heads(d)
    opt = torch.optim.Adam(heads.parameters(), lr=lr)
    batch = min(batch, len(photos))
    for epoch in range(epochs):
        perm = torch.randperm(len(photos))
        total = 0.0
        for i in range(0, len(photos), batch):
            idx = perm[i:i + batch]
            loss = game_step(heads, photos[idx], sentences[idx], temperature)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(idx)
        if verbose and (epoch % 5 == 0 or epoch == epochs - 1):
            print(f"  pass {epoch + 1:>2}/{epochs} over the pairs   loss = {total / len(photos):.2f}")
    return heads


@torch.no_grad()
def shared(heads, photos=photo_codes, sentences=title_codes):
    """Every photo and every sentence in the shared space: two tables of length-1 codes."""
    heads.eval()
    return F.normalize(heads.photo(photos), dim=1), F.normalize(heads.text(sentences), dim=1)

### 2.4 ▶ · Train: twenty passes over 4,800 pairs

About a second. Then the score, in both directions, on the 1,200 held-out
products.

In [ ]:
# Given - read it, run it, move on.
t0 = time.time()
heads = train_heads(photo_codes[train], title_codes[train], verbose=True)
print(f"trained in {time.time() - t0:.1f}s")

P, T = shared(heads)          # P: the 6,000 photos, T: the 6,000 titles, in the shared space
for k in (1, 5, 10):
    print(f"found in the top {k:>2}:   sentence -> photo {found_in_top(T[held], P[held], k):.3f}"
          f"   photo -> sentence {found_in_top(P[held], T[held], k):.3f}")

### 2.5 ▶ · The trap, on purpose

"Found in the top 10": **of which products, and out of how many?** The same
heads, the same function, four honest answers. The first is the one an AI's
first draft very often prints, because it is the easiest to compute: the score
inside one training batch.

In [ ]:
# Given - read it, run it, move on.
seen = torch.where(train)[0][:256]                         # one batch it trained on
g = torch.Generator().manual_seed(SEED)
few = torch.where(held)[0][torch.randperm(int(held.sum()), generator=g)[:256]]

trap = {
    "256 products\nit trained on": found_in_top(T[seen], P[seen]),
    "256 held-out\nproducts": found_in_top(T[few], P[few]),
    "the 1,200 held-out\nproducts": found_in_top(T[held], P[held]),
    "held-out sentences,\nall 6,000 photos": found_in_top(T[held], torch.cat([P[held], P[train]])),
}
for name, value in trap.items():
    print(f"{name.replace(chr(10), ' '):<40s} found in the top 10 = {value:.3f}")

fig, ax = plt.subplots(figsize=(7.4, 3.2))
bars = ax.bar(list(trap), list(trap.values()),
              color=[PALETTE["accent"], PALETTE["teal_dim"], PALETTE["teal"], PALETTE["teal_dim"]])
ax.bar_label(bars, fmt="{:.0%}"); ax.set_ylim(0, 1.12); ax.tick_params(axis="x", labelsize=8)
ax.set_title("Checkpoint 2 - the same heads, four honest scores")
plt.show()

### ❓ Checkpoint 2

1. What is the loss after the first pass, and after the last? `log(256)` is
   5.55: what would the heads be doing in the game if the loss stayed there?
2. Write down found in the top 1, 5 and 10, in both directions. By what factor
   did the top 10 improve on the untrained heads of checkpoint 1?
3. In 2.5 the same heads get four scores. Write them down. Which one do you
   give to the person who decides whether to ship this search, and why not the
   first?

**⭐ Going further - optional**

4. Training took about a second on a laptop. CLIP took weeks on hundreds of
   GPUs. What did we **not** have to learn?

*Your answers:*

1.
2.
3.
⭐ 4.

---

# Checkpoint 3 - Type a sentence, get products

**~15 minutes.** One cell to write, three lines.

### 3.1 ✏️ · The search

A sentence goes through the sentence expert (`sentence_code`, given), then
through **your** text head: it is now a point of the shared space. The
products to return are the ones whose photo is closest to it.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 3 lines
#     q:    sentence_code(sentence) through model.text, scaled to length 1 (dim=0)
#     sim:  photos @ q, one similarity per product
#     best: sim.topk(k)
# ──────────────────────────────────────────────────────────────────────
def search(sentence, k=5, model=None):
    """The rows of the k products closest to the sentence, and their similarities."""
    model = model or heads
    photos, _ = shared(model)                    # the 6,000 photos in the shared space
    ...          # <- replace this line
    return best.indices, best.values


rows, sims = search("a red t-shirt")
print("a red t-shirt ->", [meta["title"][int(r)] for r in rows])

### 3.2 ▶ · Four requests, then yours

The number in front of each row is the similarity of the best product.

In [ ]:
# Given - read it, run it, move on.
def ask(sentence, model=None, note=""):
    rows, sims = search(sentence, model=model)
    show_products(rows, f'"{sentence}"{note}   (similarities {sims[0]:.2f} to {sims[-1]:.2f})')


for sentence in ("a red t-shirt", "a brown leather wallet", "green flip flops", "a watch for women"):
    ask(sentence)

In [ ]:
# Given - read it, run it, move on.
# Your own request. If the sentence expert is not loaded, pick one from list(CANNED).
MY_REQUEST = "black running shoes"
ask(MY_REQUEST)

### 3.3 ▶ · When it misses, how far does it miss?

For each held-out title, look at the **first** photo returned: is it the exact
product? At least the same kind? The same kind and the same colour?

In [ ]:
# Given - read it, run it, move on.
first = (T[held] @ P[held].T).argmax(1).numpy()
h = meta[held.numpy()].reset_index(drop=True)
kind, colour = h["article_type"].to_numpy(), h["colour"].to_numpy()

print(f"the first photo is the exact product         : {(first == np.arange(len(h))).mean():.3f}")
print(f"the first photo is the same kind of product  : {(kind == kind[first]).mean():.3f}")
print(f"... the same kind and the same colour        : "
      f"{((kind == kind[first]) & (colour == colour[first])).mean():.3f}")

miss = int(np.where(first != np.arange(len(h)))[0][3])
held_rows = torch.where(held)[0]
print(f'\none miss - the title was: "{h["title"][miss]}"')
top3 = (T[held][miss] @ P[held].T).topk(3).indices
show_products([held_rows[miss]] + [held_rows[i] for i in top3],
              "left: the product we were looking for.   then: the three photos returned",
              frame=held_rows[miss])

### 3.4 ▶ · Four requests that leave the titles

A paraphrase, a negation, a product the shop does not sell, and a price.

In [ ]:
# Given - read it, run it, move on.
for sentence in ("something to carry my things", "a t-shirt that is not black",
                 "a winter coat", "a watch under 50 euros"):
    ask(sentence)

### ❓ Checkpoint 3

1. For three of the requests of 3.2, how many of the five products are right?
2. "a t-shirt that is not black": what colour are the t-shirts returned? Titles
   look like "Puma Men Black T-shirt": how often would a title contain "not"?
3. "a winter coat": the shop sells no coat. What does the search return, and
   with what similarity, compared with "a red t-shirt"? Does this search ever
   answer "nothing found"?

**⭐ Going further - optional**

4. In 3.3 the first photo is the exact product about one time in four, but the
   right kind nine times in ten. Look at the miss shown below the numbers and
   explain why "top 1" is a harsh score on this catalogue.

*Your answers:*

1.
2.
3.
⭐ 4.

---

# Checkpoint 4 - The sentences are the instruction

**~20 minutes. The most important part of the lab.** Two cells to write.

Yesterday the list of edits decided what the space ignored. Today the
**sentences** paired with the photos decide what the space can be asked. You
will measure it.

### 4.1 ✏️ · Name a product without training a classifier

Write one sentence per kind of product - "a photo of a watch", "a photo of a
handbag"... - and put the twelve sentences in the shared space. A photo is
**named** after the sentence it lands closest to. No classifier, no labels.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 3 lines
#     s: sentence_codes through model.text, scaled to length 1
#     p: photos through model.photo, scaled to length 1
#     return, for each row of p, the index of its most similar row of s (argmax)
# ──────────────────────────────────────────────────────────────────────
def name_it(model, sentence_codes, photos):
    """For each photo code, the index of the closest sentence. Nothing is trained here."""
    with torch.no_grad():
        ...          # <- replace this line

### 4.2 ▶ · Six ways of writing the twelve sentences

The sentences were encoded in advance by the sentence expert. Same heads, same
photos: only the wording changes. Then the same game with eight colours
("a black one", "a red one"...).

*(Papers: zero-shot classification. The sentences are called prompts.)*

In [ ]:
# Given - read it, run it, move on.
prompt_sets = [str(s) for s in blob["prompt_sets"]]
prompt_text = blob["prompt_text"]
prompt_codes = F.normalize(torch.from_numpy(blob["prompt_emb"]), dim=2)       # (6 sets, 12 kinds, 384)

colours = [str(c) for c in blob["colour_names"]]
colour_codes = F.normalize(torch.from_numpy(blob["colour_prompt_emb"][1]), dim=1)   # "a black one", ...
has_colour = held & torch.tensor(meta["colour"].isin(colours).to_numpy())
colour_of = torch.tensor([colours.index(c) if c in colours else -1 for c in meta["colour"]])


def kinds_named(model, wording=2):
    """Share of held-out photos given the right kind, with the sentences 'a photo of ...'."""
    pred = name_it(model, prompt_codes[wording], photo_codes[held])
    return (pred == kind_of[held]).float().mean().item()


def colours_named(model):
    """Share of held-out photos given the right colour, among eight colours."""
    pred = name_it(model, colour_codes, photo_codes[has_colour])
    return (pred == colour_of[has_colour]).float().mean().item()


for w, name in enumerate(prompt_sets):
    print(f"{name:<22s} kinds named = {kinds_named(heads, w):.3f}     e.g. "
          f'"{prompt_text[w][0]}", "{prompt_text[w][11]}"')
print(f"\ncolours named = {colours_named(heads):.3f}"
      f"   (always answering 'black' would give "
      f"{(colour_of[has_colour] == colours.index('Black')).float().mean():.3f})")

In [ ]:
# Given - read it, run it, move on.
# Where does it hesitate? Rows: the true kind. Columns: the kind it named.
pred = name_it(heads, prompt_codes[2], photo_codes[held])
pd.crosstab(pd.Series([kinds[i] for i in kind_of[held]], name="true kind"),
            pd.Series([kinds[i] for i in pred], name="named"))

### 4.3 ✏️ · One row of the table

Train a new pair of heads on (photo, **some sentences**), then ask the three
questions of today. The questions are always asked the same way: the
customer's requests are the **full titles**, whatever the heads were trained
on.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 2 lines
#     model:  train_heads on the TRAIN photos and the TRAIN rows of `sentences`
#     P_, T_: shared(model) - the photos and the full titles in its shared space
# ──────────────────────────────────────────────────────────────────────
def sentence_row(name, sentences):
    """Train heads on (photo, these sentences), measure them. Returns (row, heads)."""
    ...          # <- replace this line
    row = {"sentences paired with the photos": name,
           "found in top 10": found_in_top(T_[held], P_[held]),
           "kinds named": kinds_named(model),
           "colours named": colours_named(model)}
    return row, model


row, _ = sentence_row("the full title", title_codes)
print(row)          # sanity check: the numbers of checkpoints 2 and 4.2

### 4.4 ▶ · Four sets of sentences, a few seconds

Same photos, same heads, same twenty passes. Only the sentence paired with
each photo changes:

- **someone else's title** - the pairs are shuffled;
- **the kind only** - "Tshirts", "Watches": one of twelve words;
- **the title without its colour words**;
- **the full title**.

In [ ]:
# Given - read it, run it, move on.
nocolour_codes = load_codes("products_text_nocolour_emb.npy")
kind_only_codes = prompt_codes[0][kind_of]           # the database label of the kind, nothing else
g = torch.Generator().manual_seed(SEED)
shuffle = torch.randperm(len(title_codes), generator=g)
shuffled_codes = title_codes[shuffle]                # every photo gets another product's title

r = int(torch.where(train)[0][26])
print("one product, four sentences")
print("   someone else's title :", meta["title"][int(shuffle[r])])
print("   the kind only        :", meta["article_type"][r])
print("   without colour words :", meta["title_nocolour"][r])
print("   the full title       :", meta["title"][r])

rows, sentence_models = [], {}
for name, sentences in (("someone else's title", shuffled_codes),
                        ("the kind only", kind_only_codes),
                        ("the title without colour words", nocolour_codes),
                        ("the full title", title_codes)):
    row, sentence_models[name] = sentence_row(name, sentences)
    rows.append(row)

table = pd.DataFrame(rows).set_index("sentences paired with the photos")
table.round(3)

In [ ]:
# Given - read it, run it, move on.
# The same request, asked of two spaces.
ask("a red t-shirt", note="  -  heads trained on the full titles")
ask("a red t-shirt", model=sentence_models["the title without colour words"],
    note="  -  heads trained on titles without colour words")

### ❓ Checkpoint 4 - the questions that matter most today

`found in top 10` asks for the exact product. `kinds named` and `colours
named` ask whether the space knows the kind, and the colour, of a photo.

1. Compare the last two rows. When the colour words are removed from the
   training sentences, what happens to `colours named`? To `kinds named`?
   *(Always answering "black" gives about 0.30.)*
2. "The kind only" gives the best `kinds named` of the table and almost the
   worst `found in top 10`. Why is that not a contradiction?
3. A customer will ask for "cotton", "waterproof", "for the gym". Count the
   titles that contain each word, for example
   `meta["title"].str.contains("cotton", case=False).sum()`. What would you
   pair with the photos to make these requests work, and where would those
   sentences come from in a real company?

**⭐ Going further - optional**

4. In 4.2, the sentences written in French name almost nothing correctly. The
   photos have not changed: which of the two experts is at fault, and what
   would you replace?
5. In 4.2, "customer words" and "a photo of ..." differ by a point or so. Is
   that a real effect? *(Checkpoint 6 lets you retrain with other settings:
   how much does the number move then?)*

*Your answers:*

1.
2.
3.
⭐ 4.
⭐ 5.

---

# Checkpoint 5 - How many pairs, and which temperature

**~10 minutes.** One cell to write, three lines.

### 5.1 ▶ · The number of pairs

CLIP was trained on 400 million pairs. You have 4,800. What does each
doubling buy?

In [ ]:
# Given - read it, run it, move on.
g = torch.Generator().manual_seed(SEED)
order = torch.where(train)[0][torch.randperm(int(train.sum()), generator=g)]

pair_counts = (150, 300, 600, 1200, 2400, 4800)
pair_scores = []
for n in pair_counts:
    some = order[:n].sort().values               # n training products, picked at random
    model = train_heads(photo_codes[some], title_codes[some])
    P_, T_ = shared(model)
    pair_scores.append(found_in_top(T_[held], P_[held]))
    print(f"{n:>5} pairs   found in top 10 = {pair_scores[-1]:.3f}")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(pair_counts, pair_scores, marker="o", color=PALETTE["accent"])
ax.set_xscale("log"); ax.set_xticks(pair_counts); ax.set_xticklabels(pair_counts)
ax.set_xlabel("pairs (photo, title) used for training"); ax.set_ylabel("found in top 10")
ax.set_ylim(0, 1); ax.set_title("Checkpoint 5 - more pairs, better search")
plt.show()

### 5.2 ✏️ · The temperature

Yesterday's dial. For each temperature: train heads on the 4,800 training
pairs, put everything in the shared space, measure found in top 10 on the
held-out products.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 3 lines
#     model:  train_heads(...) on the training pairs, at temperature t
#     P_, T_: shared(model)
#     append found_in_top(T_[held], P_[held]) to temperature_scores
# ──────────────────────────────────────────────────────────────────────
temperatures = (0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0)
temperature_scores = []
for t in temperatures:
    ...          # <- replace this line

In [ ]:
# Given - read it, run it, move on.
for t, s in zip(temperatures, temperature_scores):
    print(f"temperature {t:<5}  found in top 10 = {s:.3f}")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(temperatures, temperature_scores, marker="o", color=PALETTE["teal"])
ax.set_xscale("log"); ax.set_xticks(temperatures); ax.set_xticklabels(temperatures)
ax.set_xlabel("temperature"); ax.set_ylabel("found in top 10"); ax.set_ylim(0, 1)
ax.set_title("Checkpoint 5 - the temperature dial")
plt.show()

### ❓ Checkpoint 5

1. What is found in top 10 with 150 pairs, and with 4,800? How many points did
   the last doubling (2,400 to 4,800) bring? Has the curve flattened?
2. Which temperature gives the best score? What is the score at temperature 1?
3. The shop's full catalogue has 44,000 products, each with a photo and a
   title. How many pairs is that, and what did they cost to label? From your
   curve, would you expect them to help?

*Your answers:*

1.
2.
3.

---

# Checkpoint 6 - The challenge

**Open-ended. Most of you will not get here, and that is fine.**

> Build the heads with the highest **found in top 10** on the held-out
> products, with a shared space of **64 numbers** per product, while keeping
> **`kinds named` at 0.80 or more**.

The reference is the heads of checkpoint 2: about **0.70** found in the top
10, at about **0.82** kinds named.

| Lever | Try |
|---|---|
| `temperature` | between 0.05 and 0.3 |
| `epochs` | more passes over the pairs - watch `kinds named` |
| `lr`, `batch` | the size of each nudge, the number of products in each game |
| `hidden` | a bigger head: one hidden layer of this many numbers before the 64 |
| `dropout` | hide a share of each expert's numbers during training |

**The rules.** Keep a row for **every** configuration, including the losers.
One change at a time.

### 6.1 ▶ · The harness, given

In [ ]:
# Given - read it, run it, move on.
results = []


def attempt(name, *, temperature=0.1, epochs=20, lr=1e-3, batch=256, hidden=0, dropout=0.0):
    """Train heads with these settings, record found in top 10 and kinds named."""
    set_seed(SEED)
    model = Heads(64)
    if hidden or dropout:
        def head(n_in):
            layers = [nn.Dropout(dropout)] if dropout else []
            layers += [nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, 64)] if hidden \
                else [nn.Linear(n_in, 64)]
            return nn.Sequential(*layers)
        model.photo, model.text = head(512), head(384)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    photos, sentences = photo_codes[train], title_codes[train]
    t0 = time.time()
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(photos))
        for i in range(0, len(photos), batch):
            idx = perm[i:i + batch]
            loss = game_step(model, photos[idx], sentences[idx], temperature)
            opt.zero_grad(); loss.backward(); opt.step()
    secs = time.time() - t0
    P_, T_ = shared(model)
    found, named = found_in_top(T_[held], P_[held]), kinds_named(model)
    ok = "ok " if named >= 0.80 else "OUT"
    results.append({"config": name, "seconds": round(secs, 1), "found in top 10": found,
                    "kinds named": named, "valid": named >= 0.80})
    print(f"  {ok} {name:<34s} {secs:4.1f}s   found in top 10 = {found:.3f}   kinds named = {named:.3f}")
    return model


_ = attempt("reference")

### 6.2 ✏️ · Your attempts

One `attempt(...)` per configuration. Change **one thing** at a time.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about as many lines as you have ideas
#     one attempt(...) per configuration, one change at a time
# ──────────────────────────────────────────────────────────────────────
# _ = attempt("epochs=40", epochs=40)
# _ = attempt("temperature=0.15", temperature=0.15)
# ... keep going, and keep the losers

In [ ]:
# Given - read it, run it, move on.
pd.DataFrame(results).sort_values(["valid", "found in top 10"], ascending=False)

### ❓ Checkpoint 6

1. Your best valid configuration: its `found in top 10` and its `kinds named`.
2. **Which single lever mattered most?** Which two rows of your table prove it?
3. Did a bigger head (`hidden`) help `found in top 10`? What did it do to
   `kinds named`?

**⭐ Going further - optional**

4. You chose your settings by looking at the score on the held-out products.
   Is your best score still an honest estimate of what a new product would
   get? What would you need to make it one?

*Your answers:*

1.
2.
3.
⭐ 4.

---

## What you built today

- A **shared space** for photos and sentences, from two frozen experts and two
  small heads trained in about a second - with yesterday's matching game,
  unchanged.
- A search you can **type at**: a sentence in, products out, with the right
  product in the top 10 about seven times in ten on products the heads never
  saw.
- A way to **name** a product with no classifier at all: twelve sentences, and
  the closest one wins.
- The measured rule of the day: **the space can be asked what its sentences
  said, and nothing else**. Remove the colour words and colour is gone; keep
  only the kind and only the kind is left.
- A score that reads 100 %, 94 %, 70 % or 38 % for the same heads: "top 10" is
  not a number until you say **of what, among how many**.

**Next session, the last one:** the search becomes an assistant. A language
model reads the customer's request, calls the search you built today, and
writes its answer from the products it gets back.